In [7]:
import os

try:
    import google.colab
    os.system('git clone https://github.com/ethanresek/luminal-classifiers 2>/dev/null; cd /content/luminal-classifiers && git pull')
    os.system('pip install skorch')
except ImportError:
    pass

In [8]:
import numpy as np
import pandas as pd
import sys
import joblib
import torch.nn.functional as F

from datetime import datetime
from typing import override
from torch import nn, ones, sigmoid, optim
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, StratifiedKFold
from skorch import NeuralNetBinaryClassifier
from skorch.callbacks import EarlyStopping
from sklearn.metrics import f1_score, roc_auc_score, balanced_accuracy_score

print("Imports OK")

Imports OK


In [9]:
FEATURE_COUNTS = [5, 10, 20, 50]
RANDOM_SEED = 42

In [10]:
class SparsityNNBC(NeuralNetBinaryClassifier):
    def __init__(self, **kwargs):
        super(SparsityNNBC, self).__init__(**kwargs)

    @override
    def get_loss(self, y_pred, y_true, X=None, training=False):
        sparsity_effect = self.module_.sparsity * sigmoid(self.module_.gate).sum()
        loss = super(SparsityNNBC, self).get_loss(y_pred, y_true, X, training) + sparsity_effect
        return loss


class MLP(nn.Module):
    def __init__(self, hidden_sizes, dropout, sparsity, input_size=489, output_size=1):
        super(MLP, self).__init__()

        self.sparsity = sparsity
        self.gate = nn.Parameter(ones(input_size))

        sizes = [input_size] + hidden_sizes + [output_size]
        self.linears = nn.ModuleList([
            nn.Linear(sizes[i], sizes[i + 1]) for i in range(len(sizes) - 1)
        ])
        self.dropout = nn.Dropout(dropout)
        self.batch_norms = nn.ModuleList([
            nn.BatchNorm1d(h) for h in hidden_sizes
        ])

    def forward(self, x):
        x = x * sigmoid(self.gate)

        for i, l in enumerate(self.linears[:-1]):
            x = l(x)
            x = self.batch_norms[i](x)
            x = F.relu(x)
            x = self.dropout(x)

        x = self.linears[-1](x)

        return x

In [11]:
try:
    from google.colab import drive
    sys.path.append('/content/luminal-classifiers')
    file_name = '/content/luminal-classifiers/models/tuned_MLP_20260415_001159.joblib'
    CSV = '/content/luminal-classifiers/data/METABRIC_RNA_Mutation.csv'
    print('Working in Colab')
except ImportError:
    CSV = 'data/METABRIC_RNA_Mutation.csv'
    file_name = 'models/tuned_MLP_20260415_001159.joblib'
    print('Working locally')
finally:
    from pre_process import preprocess
    data = joblib.load(file_name)
    model = data['model']
    X_train = data['X_train']
    y_train = data['y_train']
    X_test = data['X_test']
    y_test = data['y_test']

Working locally


In [12]:
DF = pd.read_csv(CSV, low_memory=False)
Y_OLD_NAME = 'pam50_+_claudin-low_subtype'
KEEP = list(DF.columns[31:520]) + [Y_OLD_NAME]

X, y = preprocess(DF, KEEP)

In [14]:
gate = model.module_.gate
s_gate = sigmoid(gate).detach().numpy()
feature_names = X.columns
all_top_k_idx = []

for k in FEATURE_COUNTS:
    top_k_idx = np.argsort(s_gate)[-k:]
    all_top_k_idx.append(top_k_idx)
    top_k_names = feature_names[top_k_idx]
    print(f"Top {k}: {list(top_k_names)}")

Top 5: ['aurka', 'mmp13', 'mmp25', 'mmp10', 'bard1']
Top 10: ['flt3', 'mapk8', 'mmp7', 'fancd2', 'cdk2', 'aurka', 'mmp13', 'mmp25', 'mmp10', 'bard1']
Top 20: ['clk3', 'brca1', 'cdk1', 'cdc25a', 'serpini1', 'e2f2', 'pik3r1', 'cacna2d3', 'aph1b', 'tg', 'flt3', 'mapk8', 'mmp7', 'fancd2', 'cdk2', 'aurka', 'mmp13', 'mmp25', 'mmp10', 'bard1']
Top 50: ['hsd17b6', 'gldc', 'rad51', 'map3k1', 'ugt2b7', 'cyp3a5', 'acvr1b', 'hsd17b8', 'mapk14', 'sbno1', 'map3k3', 'kit', 'lamb3', 'hes6', 'pik3r2', 'tgfbr3', 'mmp15', 'tgfb3', 'lipi', 'prkcq', 'ccnb1', 'zfyve9', 'rps6kb2', 'e2f7', 'stat5b', 'ppp2r2a', 'spry2', 'rad51c', 'pik3r3', 'nr2f1', 'clk3', 'brca1', 'cdk1', 'cdc25a', 'serpini1', 'e2f2', 'pik3r1', 'cacna2d3', 'aph1b', 'tg', 'flt3', 'mapk8', 'mmp7', 'fancd2', 'cdk2', 'aurka', 'mmp13', 'mmp25', 'mmp10', 'bard1']


In [ ]:
searches = []

for arr in all_top_k_idx:
    print("------------------------------------------------------")
    print(f"Searching for hyperparameters for {len(arr)} features")
    print("------------------------------------------------------")
    X_train_k = X_train[:, arr]
    X_test_k = X_test[:, arr]

    net = SparsityNNBC(
        module=MLP,
        criterion=nn.BCEWithLogitsLoss,
        optimizer=optim.Adam,
        max_epochs=100,
        callbacks=[EarlyStopping(patience=10)],
        module__hidden_sizes=[128, 64],
        module__dropout=0.3,
        module__input_size=len(arr),
        module__sparsity=1e-3,
        optimizer__weight_decay=1e-3,
        verbose=0
    )

    p_dist = {
        'module__hidden_sizes': [[256, 128], [128, 64], [64, 32], [128]],
        'module__dropout': [0.1, 0.2, 0.3, 0.4, 0.5],
        'module__sparsity': [1e-4, 1e-3, 5e-3, 1e-2],
        'optimizer__weight_decay': [1e-4, 1e-3, 1e-2],
        'lr': [1e-4, 5e-4, 1e-3, 5e-3, 1e-2],
        'batch_size': [16, 32, 64],
    }

    rand_search = RandomizedSearchCV(net, param_distributions=p_dist, scoring='f1', n_iter=100, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED), random_state=RANDOM_SEED)
    rand_search.fit(X_train_k, y_train)

    y_pred = rand_search.predict(X_test_k)
    y_pred_prob = rand_search.predict_proba(X_test_k)[:, 1]

    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    balanced_accuracy = balanced_accuracy_score(y_test, y_pred)

    print("Accuracies for Randomized Search CV")
    print(f"F1-score: {f1}")
    print(f"ROC-AUC: {roc_auc}")
    print(f"Balanced Accuracy: {balanced_accuracy}")

    best = rand_search.best_params_

    grid_params = {}
    for key, val in best.items():
        if (isinstance(val, str)
                or val is None
                or key == 'module__hidden_sizes'
                or key == 'batch_size'):
            grid_params[key] = [val]
        elif isinstance(val, int):
            step = max(1, round(0.1 * val))
            grid_params[key] = [max(1, val - step), val, val + step]
        elif isinstance(val, float):
            step = max(0.005, round(0.1 * val, 4))
            grid_params[key] = [max(0, round(val - step, 4)), val, val + step]

    post_rand_net = rand_search.best_estimator_

    grid_search = GridSearchCV(post_rand_net, param_grid=grid_params, scoring='f1', cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED))
    grid_search.fit(X_train_k, y_train)

    y_pred = grid_search.predict(X_test_k)
    y_pred_prob = grid_search.predict_proba(X_test_k)[:, 1]

    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    balanced_accuracy = balanced_accuracy_score(y_test, y_pred)

    print("Accuracies for Grid Search CV")
    print(f"F1-score: {f1}")
    print(f"ROC-AUC: {roc_auc}")
    print(f"Balanced Accuracy: {balanced_accuracy}")

    searches.append(grid_search)

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
os.makedirs('models', exist_ok=True)
joblib.dump({
    'searches': searches,
    'feature_indices': all_top_k_idx,
    'feature_counts': FEATURE_COUNTS
}, f"models/MLP_grid_searches_{timestamp}.joblib")